# Retos · Puzles

Puzles al estilo de [Advent of Code](https://adventofcode.com/): cada uno tiene una entrada pequeña escrita en el propio notebook y, a menudo, una segunda parte que obliga a cambiar de idea. Se mezclan rejillas, expresiones regulares, números complejos, intérpretes, búsquedas en grafos de estados, detección de ciclos y fuerza bruta con `itertools`.

In [1]:
import math
import random
import re
from collections import Counter, deque
from itertools import permutations

## 1. Rocas que ruedan

En la plataforma, `O` son rocas redondas, `#` rocas fijas y `.` suelo. Inclinarla hacia el norte para que todas las `O` rueden hacia arriba hasta chocar con el borde, un `#` u otra `O`, y calcular la carga: cada `O` aporta el número de filas que hay desde ella hasta el borde sur, incluida la suya. Hacerlo en una pasada por columna, recordando la primera fila libre, y comprobarlo contra una simulación que mueve las rocas una casilla cada vez hasta que no se mueve ninguna.

In [2]:
PLATAFORMA = '''
OO#O..O.
#O...O.O
...#OOO.
#....#..
OO.OO.O.
O#..O.#.
.#O.O.O.
O.O....O
'''.split()

def inclina_norte(rejilla):
    filas = [list(fila) for fila in rejilla]
    for c in range(len(filas[0])):
        libre = 0  # primera fila a la que puede llegar una roca de esta columna
        for f in range(len(filas)):
            if filas[f][c] == "#":
                libre = f + 1
            elif filas[f][c] == "O":
                filas[f][c] = "."
                filas[libre][c] = "O"
                libre += 1
    return ["".join(fila) for fila in filas]

def inclina_paso_a_paso(rejilla):
    filas = [list(fila) for fila in rejilla]
    movida = True
    while movida:
        movida = False
        for f in range(1, len(filas)):
            for c in range(len(filas[0])):
                if filas[f][c] == "O" and filas[f - 1][c] == ".":
                    filas[f - 1][c], filas[f][c] = "O", "."
                    movida = True
    return ["".join(fila) for fila in filas]

def carga(rejilla):
    return sum(fila.count("O") * (len(rejilla) - f) for f, fila in enumerate(rejilla))

inclinada = inclina_norte(PLATAFORMA)
for antes, despues in zip(PLATAFORMA, inclinada):
    print(antes, "  ", despues)
print("carga:", carga(PLATAFORMA), "->", carga(inclinada))

assert inclinada == inclina_paso_a_paso(PLATAFORMA)
random.seed(1)
for _ in range(200):
    rejilla = ["".join(random.choices(".O#", weights=[5, 3, 2], k=7)) for _ in range(6)]
    assert inclina_norte(rejilla) == inclina_paso_a_paso(rejilla)

OO#O..O.    OO#OOOOO
#O...O.O    #OO.OOOO
...#OOO.    .OO#O.O.
#....#..    #..OO#..
OO.OO.O.    O.......
O#..O.#.    O#....#.
.#O.O.O.    O#....O.
O.O....O    ........
carga: 106 -> 143


## 2. Instrucciones corruptas

La memoria de un programa está corrupta: entre mucha basura hay instrucciones `mul(a,b)` válidas, con `a` y `b` de 1 a 3 cifras y sin espacios. Parte 1: extraer con una expresión regular todas las válidas y sumar sus productos. Parte 2: las instrucciones `do()` y `don't()` activan y desactivan los `mul` que vienen detrás (al principio están activos). Usar una sola expresión con alternativas y `re.finditer`.

In [3]:
MEMORIA = (
    "@mul(3,4)#mul(5,6]don't()?mul(2,9)xmul(1000,2)do()mul(10,10)"
    "<mul ( 2,2)>select(7,8)mul(7,8)don't()mul(4,4)!do()&mul(123,2)mul(4,5,6)"
)
PATRON = re.compile(r"mul\((\d{1,3}),(\d{1,3})\)|do\(\)|don't\(\)")

def suma_productos(memoria, con_condiciones=False):
    activo, total = True, 0
    for m in PATRON.finditer(memoria):
        if m[0] == "do()":
            activo = True
        elif m[0] == "don't()":
            activo = False
        elif activo or not con_condiciones:
            total += int(m[1]) * int(m[2])
    return total

print([m[0] for m in PATRON.finditer(MEMORIA)])
parte1, parte2 = suma_productos(MEMORIA), suma_productos(MEMORIA, con_condiciones=True)
print("parte 1:", parte1, " parte 2:", parte2)
assert parte1 == 3 * 4 + 2 * 9 + 10 * 10 + 7 * 8 + 4 * 4 + 123 * 2
assert parte2 == 3 * 4 + 10 * 10 + 7 * 8 + 123 * 2

['mul(3,4)', "don't()", 'mul(2,9)', 'do()', 'mul(10,10)', 'mul(7,8)', "don't()", 'mul(4,4)', 'do()', 'mul(123,2)']
parte 1: 448  parte 2: 414


## 3. Paseo con números complejos

Desde el origen y mirando al norte, seguir instrucciones como `R2, L3`: girar a la derecha (`R`) o a la izquierda (`L`) y avanzar ese número de manzanas. Representar la posición y la dirección con números complejos: girar a la izquierda es multiplicar por $i$ (`1j`) y a la derecha por $-i$. Parte 1: distancia Manhattan al punto final. Parte 2: guardando las posiciones visitadas en un conjunto, distancia a la primera manzana por la que se pasa dos veces. Comprobarlo con una versión que usa tuplas y una lista de direcciones.

In [4]:
INSTRUCCIONES = "R5, L3, L2, L6, R2, R4, L1, L8, R3, R9"

def paseo(instrucciones):
    posicion, direccion = 0, 1j  # norte
    visitadas, repetida = {posicion}, None
    for giro, pasos in re.findall(r"([LR])(\d+)", instrucciones):
        direccion *= 1j if giro == "L" else -1j
        for _ in range(int(pasos)):
            posicion += direccion
            if posicion in visitadas and repetida is None:
                repetida = posicion
            visitadas.add(posicion)
    return posicion, repetida

def paseo_tuplas(instrucciones):
    rumbos = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # N, E, S, O en sentido horario
    x = y = rumbo = 0
    camino = [(0, 0)]
    for giro, pasos in re.findall(r"([LR])(\d+)", instrucciones):
        rumbo = (rumbo + (1 if giro == "R" else -1)) % 4
        for _ in range(int(pasos)):
            x, y = x + rumbos[rumbo][0], y + rumbos[rumbo][1]
            camino.append((x, y))
    repetida = next((p for i, p in enumerate(camino) if p in camino[:i]), None)
    return (x, y), repetida

def manhattan(z):
    return int(abs(z.real) + abs(z.imag))

final, repetida = paseo(INSTRUCCIONES)
print("final:", final, "distancia", manhattan(final))
print("primera repetida:", repetida, "distancia", manhattan(repetida))

random.seed(3)
for _ in range(200):
    instrucciones = ", ".join(random.choice("LR") + str(random.randint(1, 6)) for _ in range(random.randint(1, 12)))
    z, r = paseo(instrucciones)
    (x, y), rt = paseo_tuplas(instrucciones)
    assert (z.real, z.imag) == (x, y)
    assert (r is None and rt is None) or (r.real, r.imag) == rt

final: (-3+2j) distancia 5
primera repetida: (3+0j) distancia 3


## 4. Máquina virtual

Escribir un intérprete para un ensamblador con cuatro registros (`a` a `d`, a cero al empezar) e instrucciones `cpy x r` (copia en `r` un número o un registro), `inc r`, `dec r` y `jnz x d` (salta `d` instrucciones si `x` no es cero). Separar cada línea con `split` y ejecutarla con `match`, contando las instrucciones ejecutadas. Probarlo con un programa que multiplica `b * c` con dos bucles anidados y otro que calcula el Fibonacci $c$-ésimo.

In [5]:
def valor(x, registros):
    return registros[x] if x in registros else int(x)

def ejecuta(programa, **iniciales):
    instrucciones = [linea.split(";")[0].split() for linea in programa.strip().splitlines()]
    registros = dict.fromkeys("abcd", 0) | iniciales
    pc = pasos = 0
    while 0 <= pc < len(instrucciones):
        pasos += 1
        match instrucciones[pc]:
            case ["cpy", x, r]:
                registros[r] = valor(x, registros)
            case ["inc", r]:
                registros[r] += 1
            case ["dec", r]:
                registros[r] -= 1
            case ["jnz", x, d] if valor(x, registros) != 0:
                pc += valor(d, registros)
                continue
            case ["jnz", _, _]:
                pass
            case otra:
                raise ValueError(f"instrucción desconocida: {otra}")
        pc += 1
    return registros, pasos

MULTIPLICA = '''
cpy 0 a
cpy c d     ; d = c
inc a       ; a += 1, c veces
dec d
jnz d -2
dec b
jnz b -5    ; repite b veces
'''

FIBONACCI = '''
cpy 0 a     ; a = F(0)
cpy 1 b     ; b = F(1)
cpy a d
cpy b a     ; a = b
jnz d 2
jnz 1 4     ; si d es 0 se salta la suma
inc b       ; b += d
dec d
jnz d -2
dec c
jnz c -8
'''

registros, pasos = ejecuta(MULTIPLICA, b=6, c=7)
print(registros, pasos, "instrucciones")
for b in range(1, 7):
    for c in range(1, 7):
        assert ejecuta(MULTIPLICA, b=b, c=c)[0]["a"] == b * c

fib = [0, 1]
while len(fib) < 25:
    fib.append(fib[-1] + fib[-2])
for n in range(1, 21):
    registros, pasos = ejecuta(FIBONACCI, c=n)
    assert registros["a"] == fib[n]
    if n % 5 == 0:
        print(f"F({n}) = {registros['a']} en {pasos} instrucciones")

{'a': 42, 'b': 0, 'c': 7, 'd': 0} 145 instrucciones
F(5) = 5 en 49 instrucciones
F(10) = 55 en 317 instrucciones
F(15) = 610 en 3036 instrucciones
F(20) = 6765 en 32938 instrucciones


## 5. Torres de Hanói con pasos contados

Las torres de Hanói ya se resolvieron con recursión en `09-recursividad`. Ahora, con un generador de movimientos, comprobar que hacen falta $2^n - 1$ pasos y, sin generarlos, calcular el movimiento $k$-ésimo y en qué varilla está cada disco tras $k$ movimientos. La idea: el disco mayor se mueve en el paso $2^{n-1}$; antes, los $n-1$ menores van de la varilla origen a la auxiliar, y después, de la auxiliar al destino. Bajando disco a disco se responde en $O(n)$, también para $n = 64$ y $k = 10^{18}$.

In [6]:
def hanoi(n, origen="A", destino="C", auxiliar="B"):
    if n:
        yield from hanoi(n - 1, origen, auxiliar, destino)
        yield n, origen, destino
        yield from hanoi(n - 1, auxiliar, destino, origen)

def movimiento(n, k, origen="A", destino="C", auxiliar="B"):
    mitad = 2 ** (n - 1)
    if k == mitad:
        return n, origen, destino
    if k < mitad:
        return movimiento(n - 1, k, origen, auxiliar, destino)
    return movimiento(n - 1, k - mitad, auxiliar, destino, origen)

def estado(n, k, origen="A", destino="C", auxiliar="B"):
    varillas = {}
    for disco in range(n, 0, -1):
        mitad = 2 ** (disco - 1)
        if k < mitad:
            varillas[disco] = origen
            destino, auxiliar = auxiliar, destino
        else:
            varillas[disco] = destino
            k -= mitad
            origen, auxiliar = auxiliar, origen
    return [varillas[disco] for disco in range(1, n + 1)]  # del disco pequeño al grande

for k, (disco, de, a) in enumerate(hanoi(3), 1):
    print(f"{k}: disco {disco} de {de} a {a}  ->  {estado(3, k)}")

for n in range(1, 11):
    posiciones = ["A"] * n
    k = 0
    for k, mov in enumerate(hanoi(n), 1):
        disco, de, a = mov
        assert posiciones[disco - 1] == de and movimiento(n, k) == mov
        posiciones[disco - 1] = a
        assert estado(n, k) == posiciones
    assert k == 2 ** n - 1 and posiciones == ["C"] * n

k = 10 ** 18
print("n = 64, movimiento", k, ":", movimiento(64, k))
print("varillas:", "".join(estado(64, k)))
print(f"a un movimiento por segundo: {(2 ** 64 - 1) / (365.25 * 24 * 3600):.2e} años")

1: disco 1 de A a C  ->  ['C', 'A', 'A']
2: disco 2 de A a B  ->  ['C', 'B', 'A']
3: disco 1 de C a B  ->  ['B', 'B', 'A']
4: disco 3 de A a C  ->  ['B', 'B', 'C']
5: disco 1 de B a A  ->  ['A', 'B', 'C']
6: disco 2 de B a C  ->  ['A', 'C', 'C']
7: disco 1 de A a C  ->  ['C', 'C', 'C']
n = 64, movimiento 1000000000000000000 : (19, 'A', 'B')
varillas: CCCCCCCCCCCCCCCCCCBAABBACCCBBCABBBCCBBCABCCBAABCAAAAABBBBACCAAAA
a un movimiento por segundo: 5.85e+11 años


## 6. Cuadrado mágico

Encontrar todos los cuadrados mágicos $3 \times 3$ con los números del 1 al 9 (filas, columnas y diagonales suman 15) probando las $9! = 362\,880$ permutaciones, y comprobar que solo hay 8, que son los giros y reflejos de uno solo. Después, construir uno de cualquier orden impar con el método siamés: empezar en el centro de la fila de arriba y avanzar en diagonal hacia arriba a la derecha (dando la vuelta por los bordes), bajando una casilla si la siguiente ya está ocupada; es $O(n^2)$.

In [7]:
def es_magico(c):
    n = len(c)
    objetivo = n * (n * n + 1) // 2
    sumas = [sum(fila) for fila in c] + [sum(columna) for columna in zip(*c)]
    sumas += [sum(c[i][i] for i in range(n)), sum(c[i][n - 1 - i] for i in range(n))]
    return all(s == objetivo for s in sumas) and sorted(x for fila in c for x in fila) == list(range(1, n * n + 1))

def gira(m):
    return tuple(zip(*m[::-1]))

def simetrias(m):
    m = tuple(map(tuple, m))
    resultado = set()
    for _ in range(4):
        m = gira(m)
        resultado |= {m, m[::-1]}
    return resultado

magicos = [(p[0:3], p[3:6], p[6:9]) for p in permutations(range(1, 10)) if es_magico([p[0:3], p[3:6], p[6:9]])]
print(len(magicos), "cuadrados mágicos 3 x 3; el primero:")
for fila in magicos[0]:
    print(*fila)
assert len(magicos) == 8 and set(magicos) == simetrias(magicos[0])

def siames(n):
    c = [[0] * n for _ in range(n)]
    f, col = 0, n // 2
    for k in range(1, n * n + 1):
        c[f][col] = k
        sf, sc = (f - 1) % n, (col + 1) % n
        if c[sf][sc]:
            sf, sc = (f + 1) % n, col
        f, col = sf, sc
    return c

for fila in siames(5):
    print(" ".join(f"{x:2}" for x in fila))
assert all(es_magico(siames(n)) for n in range(1, 30, 2))

8 cuadrados mágicos 3 x 3; el primero:
2 7 6
9 5 1
4 3 8
17 24  1  8 15
23  5  7 14 16
 4  6 13 20 22
10 12 19 21  3
11 18 25  2  9


## 7. Ocho reinas por permutaciones

Colocar 8 reinas en un tablero sin que se ataquen (en `09-recursividad` se resolvió con *backtracking*). Como no puede haber dos en la misma fila ni columna, cada candidata es una permutación `p` con la reina de la fila `i` en la columna `p[i]`: basta recorrer las $8! = 40\,320$ permutaciones y quedarse con las que tienen todas las diagonales distintas, `len({i + p[i]}) == len({i - p[i]}) == 8`. Comprobar que hay 92 soluciones y que, reutilizando `simetrias`, solo 12 son esencialmente distintas.

In [8]:
def reinas(n):
    return [p for p in permutations(range(n))
            if len({i + c for i, c in enumerate(p)}) == n == len({i - c for i, c in enumerate(p)})]

def tablero(p):
    return tuple(tuple(int(c == p[i]) for c in range(len(p))) for i in range(len(p)))

soluciones = reinas(8)
distintas = {min(simetrias(tablero(p))) for p in soluciones}
print(len(soluciones), "soluciones,", len(distintas), "distintas salvo giros y reflejos")
print([len(reinas(n)) for n in range(1, 9)])
assert len(soluciones) == 92 and len(distintas) == 12
assert [len(reinas(n)) for n in range(1, 9)] == [1, 0, 0, 2, 10, 4, 40, 92]

for fila in tablero(soluciones[0]):
    print(" ".join("♛" if x else "·" for x in fila))

92 soluciones, 12 distintas salvo giros y reflejos
[1, 0, 0, 2, 10, 4, 40, 92]
♛ · · · · · · ·
· · · · ♛ · · ·
· · · · · · · ♛
· · · · · ♛ · ·
· · ♛ · · · · ·
· · · · · · ♛ ·
· ♛ · · · · · ·
· · · ♛ · · · ·


## 8. Criptoaritmética

Resolver SEND + MORE = MONEY: asignar a cada letra una cifra distinta, sin ceros a la izquierda, para que la suma sea correcta. Con fuerza bruta e `itertools.permutations` se prueban $10!/2! = 1\,814\,400$ asignaciones de cifras a las 8 letras. Para que cada prueba sea barata, convertir la ecuación en $\sum_\ell c_\ell \, d_\ell = 0$, donde el coeficiente $c_\ell$ de cada letra suma su peso posicional $10^k$ en los sumandos y lo resta en el resultado. Escribir un resolutor genérico, comprobar las soluciones con `eval` y usarlo también con otro acertijo.

In [9]:
from operator import mul

def resuelve(acertijo):
    izquierda, derecha = acertijo.replace(" ", "").split("=")
    palabras = [(p, 1) for p in izquierda.split("+")] + [(derecha, -1)]
    letras = sorted({letra for palabra, _ in palabras for letra in palabra})
    coeficientes = dict.fromkeys(letras, 0)
    for palabra, signo in palabras:
        for k, letra in enumerate(reversed(palabra)):
            coeficientes[letra] += signo * 10 ** k
    pesos = [coeficientes[letra] for letra in letras]
    iniciales = {palabra[0] for palabra, _ in palabras}
    soluciones = []
    for cifras in permutations(range(10), len(letras)):
        if sum(map(mul, pesos, cifras)) == 0:
            asignacion = dict(zip(letras, cifras))
            if all(asignacion[letra] for letra in iniciales):
                soluciones.append(asignacion)
    return soluciones

def sustituye(acertijo, asignacion):
    return acertijo.translate(str.maketrans({letra: str(d) for letra, d in asignacion.items()}))

for acertijo in ["SEND + MORE = MONEY", "TWO + TWO = FOUR"]:
    soluciones = resuelve(acertijo)
    print(acertijo, "->", len(soluciones), "soluciones")
    for asignacion in soluciones:
        cuenta = sustituye(acertijo, asignacion)
        assert eval(cuenta.replace("=", "=="))
        print("   ", cuenta)

assert [sustituye("SEND + MORE = MONEY", s) for s in resuelve("SEND + MORE = MONEY")] == ["9567 + 1085 = 10652"]

SEND + MORE = MONEY -> 1 soluciones
    9567 + 1085 = 10652
TWO + TWO = FOUR -> 7 soluciones
    734 + 734 = 1468
    765 + 765 = 1530
    836 + 836 = 1672
    846 + 846 = 1692
    867 + 867 = 1734
    928 + 928 = 1856
    938 + 938 = 1876


## 9. Jarras de agua con BFS

Con una jarra de 3 litros y otra de 5, sin marcas, medir exactamente 4 litros. Cada estado es la pareja `(a, b)` de litros en cada jarra y las acciones son llenar una jarra, vaciarla o verter una en otra hasta que la primera se vacíe o la segunda se llene. Una BFS sobre el grafo de estados, guardando de dónde se llegó a cada uno, da la solución con menos pasos. Generalizar a capacidades cualesquiera y comprobar que se puede medir $x$ si y solo si $x$ es múltiplo del $\gcd$ de las capacidades y no mayor que la jarra grande.

In [10]:
def sucesores(estado, capacidades):
    a, b = estado
    ca, cb = capacidades
    a_b, b_a = min(a, cb - b), min(b, ca - a)
    return {
        "llenar A": (ca, b), "llenar B": (a, cb),
        "vaciar A": (0, b), "vaciar B": (a, 0),
        "A -> B": (a - a_b, b + a_b), "B -> A": (a + b_a, b - b_a),
    }

def jarras(capacidades, objetivo):
    inicio = (0, 0)
    padres = {inicio: None}
    cola = deque([inicio])
    while cola:
        estado = cola.popleft()
        if objetivo in estado:
            camino = []
            while padres[estado]:
                anterior, accion = padres[estado]
                camino.append((accion, estado))
                estado = anterior
            return camino[::-1]
        for accion, siguiente in sucesores(estado, capacidades).items():
            if siguiente not in padres:
                padres[siguiente] = (estado, accion)
                cola.append(siguiente)
    return None

solucion = jarras((3, 5), 4)
for accion, (a, b) in solucion:
    print(f"{accion:<9} A = {a}, B = {b}")
assert len(solucion) == 6

for ca in range(1, 9):
    for cb in range(1, 9):
        for objetivo in range(10):
            posible = objetivo % math.gcd(ca, cb) == 0 and objetivo <= max(ca, cb)
            assert (jarras((ca, cb), objetivo) is not None) == posible
print("(4, 6) -> 3 litros:", jarras((4, 6), 3))

llenar B  A = 0, B = 5
B -> A    A = 3, B = 2
vaciar A  A = 0, B = 2
B -> A    A = 2, B = 0
llenar B  A = 2, B = 5
B -> A    A = 3, B = 4
(4, 6) -> 3 litros: None


## 10. Rejilla de riesgo con Dijkstra

Cada casilla de la rejilla tiene un riesgo del 1 al 9 que se paga al entrar en ella. Parte 1: riesgo mínimo de la esquina superior izquierda a la inferior derecha moviéndose en las 4 direcciones, con Dijkstra y `heapq`. Parte 2: un carro no puede avanzar más de 3 casillas seguidas en la misma dirección ni dar media vuelta, así que el estado ya no es la casilla sino `(fila, columna, dirección, pasos rectos)`. Escribir un Dijkstra genérico sobre estados y comprobar las dos partes con una relajación tipo Bellman-Ford, que repite hasta que ninguna distancia mejora.

In [11]:
import heapq

RIESGOS = [[int(x) for x in fila] for fila in '''
11111112
93949311
39219912
21271941
14319119
93814291
12591911
13181211
'''.split()]
FILAS, COLUMNAS = len(RIESGOS), len(RIESGOS[0])
RUMBOS = [(0, 1), (1, 0), (0, -1), (-1, 0)]

def dijkstra(inicio, vecinos, es_final):
    distancias = {inicio: 0}
    cola = [(0, inicio)]
    while cola:
        d, estado = heapq.heappop(cola)
        if es_final(estado):
            return d
        if d > distancias[estado]:
            continue
        for coste, siguiente in vecinos(estado):
            if d + coste < distancias.get(siguiente, math.inf):
                distancias[siguiente] = d + coste
                heapq.heappush(cola, (d + coste, siguiente))
    return math.inf

def relajacion(inicio, vecinos, es_final):
    estados, cola = {inicio}, deque([inicio])
    while cola:
        for _, siguiente in vecinos(cola.popleft()):
            if siguiente not in estados:
                estados.add(siguiente)
                cola.append(siguiente)
    distancias = dict.fromkeys(estados, math.inf)
    distancias[inicio] = 0
    cambiado = True
    while cambiado:
        cambiado = False
        for estado in estados:
            for coste, siguiente in vecinos(estado):
                if distancias[estado] + coste < distancias[siguiente]:
                    distancias[siguiente] = distancias[estado] + coste
                    cambiado = True
    return min(d for estado, d in distancias.items() if es_final(estado))

def dentro(f, c):
    return 0 <= f < FILAS and 0 <= c < COLUMNAS

def vecinos_casilla(estado):
    f, c = estado
    return [(RIESGOS[f + df][c + dc], (f + df, c + dc)) for df, dc in RUMBOS if dentro(f + df, c + dc)]

def vecinos_carro(estado):
    f, c, df, dc, rectos = estado
    resultado = []
    for nf, nc in RUMBOS:
        if (nf, nc) == (-df, -dc) or not dentro(f + nf, c + nc):
            continue
        n = rectos + 1 if (nf, nc) == (df, dc) else 1
        if n <= 3:
            resultado.append((RIESGOS[f + nf][c + nc], (f + nf, c + nc, nf, nc, n)))
    return resultado

def es_final(estado):
    return estado[:2] == (FILAS - 1, COLUMNAS - 1)

parte1 = dijkstra((0, 0), vecinos_casilla, es_final)
parte2 = dijkstra((0, 0, 0, 0, 0), vecinos_carro, es_final)
print("parte 1:", parte1, " parte 2:", parte2)
assert parte1 == relajacion((0, 0), vecinos_casilla, es_final)
assert parte2 == relajacion((0, 0, 0, 0, 0), vecinos_carro, es_final)
assert parte2 >= parte1

parte 1: 23  parte 2: 27


## 11. Ciclos de giro y $10^9$ pasos

Un ciclo de giro inclina la plataforma del ejercicio 1 hacia el norte, el oeste, el sur y el este, por ese orden (basta girar la rejilla 90° en sentido horario después de cada inclinación al norte). Calcular la carga tras $10^9$ ciclos: simularlos es imposible, pero como el número de estados es finito la secuencia acaba repitiéndose. Guardar en un diccionario en qué ciclo se vio cada estado; cuando uno se repite, con el inicio $\mu$ y la longitud $\lambda$ del ciclo, el estado tras $N$ ciclos es el mismo que tras $\mu + (N - \mu) \bmod \lambda$. Comprobar la extrapolación contra la simulación directa para $N$ pequeños.

In [12]:
def gira_horario(rejilla):
    return ["".join(fila) for fila in zip(*rejilla[::-1])]

def ciclo_de_giro(rejilla):
    for _ in range(4):
        rejilla = gira_horario(inclina_norte(rejilla))
    return rejilla

def tras_ciclos(rejilla, n):
    vistos, historial = {}, []
    for i in range(n):
        if tuple(rejilla) in vistos:
            mu = vistos[tuple(rejilla)]
            return historial[mu + (n - mu) % (i - mu)], mu, i - mu
        vistos[tuple(rejilla)] = i
        historial.append(rejilla)
        rejilla = ciclo_de_giro(rejilla)
    return rejilla, None, None

def simula(rejilla, n):
    for _ in range(n):
        rejilla = ciclo_de_giro(rejilla)
    return rejilla

for antes, despues in zip(PLATAFORMA, ciclo_de_giro(PLATAFORMA)):
    print(antes, "  ", despues)

final, mu, lam = tras_ciclos(PLATAFORMA, 10 ** 9)
print(f"el ciclo empieza en {mu} y dura {lam}; carga tras 10^9 ciclos: {carga(final)}")

for n in range(60):
    assert tras_ciclos(PLATAFORMA, n)[0] == simula(PLATAFORMA, n)

OO#O..O.    .O#...OO
#O...O.O    #....OOO
...#OOO.    .OO#...O
#....#..    #...O#.O
OO.OO.O.    ......OO
O#..O.#.    O#..OO#.
.#O.O.O.    O#....OO
O.O....O    ....OOOO
el ciclo empieza en 3 y dura 7; carga tras 10^9 ciclos: 89


## 12. Polímeros que crecen

Partiendo de una plantilla como `ABCA`, en cada paso se inserta entre cada pareja de letras adyacentes la letra que indica su regla (`AB -> C` convierte `AB` en `ACB`). Parte 1: tras 10 pasos, construir la cadena y restar las apariciones de la letra menos común a las de la más común. Parte 2: tras 40 pasos la cadena tendría billones de letras, porque su longitud casi se duplica en cada paso, así que en vez de la cadena se cuentan las parejas con un `Counter`: cada pareja `AB` con regla `C` se convierte en `AC` y `CB`. El coste pasa de $O(2^p)$ a $O(p \cdot r)$, con $p$ pasos y $r$ reglas.

In [13]:
PLANTILLA = "ABCA"
REGLAS = {"AA": "B", "AB": "C", "AC": "A", "BA": "A", "BB": "C", "BC": "B", "CA": "C", "CB": "A", "CC": "B"}

def crece(polimero, reglas):
    return polimero[0] + "".join(reglas.get(a + b, "") + b for a, b in zip(polimero, polimero[1:]))

def cuenta_letras(plantilla, reglas, pasos):
    parejas = Counter(a + b for a, b in zip(plantilla, plantilla[1:]))
    for _ in range(pasos):
        nuevas = Counter()
        for pareja, veces in parejas.items():
            if pareja in reglas:
                nuevas[pareja[0] + reglas[pareja]] += veces
                nuevas[reglas[pareja] + pareja[1]] += veces
            else:
                nuevas[pareja] += veces
        parejas = nuevas
    letras = Counter({plantilla[-1]: 1})  # la última letra no es la primera de ninguna pareja
    for pareja, veces in parejas.items():
        letras[pareja[0]] += veces
    return letras

def diferencia(letras):
    return max(letras.values()) - min(letras.values())

polimero = PLANTILLA
for paso in range(1, 13):
    polimero = crece(polimero, REGLAS)
    if paso <= 3:
        print(paso, polimero)
    assert Counter(polimero) == cuenta_letras(PLANTILLA, REGLAS, paso)

print("parte 1:", diferencia(cuenta_letras(PLANTILLA, REGLAS, 10)))
letras = cuenta_letras(PLANTILLA, REGLAS, 40)
print("parte 2:", diferencia(letras), "con", f"{letras.total():,}", "letras")
assert letras.total() == 3 * 2 ** 40 + 1

1 ACBBCCA
2 AACABCBBCBCCA
3 ABAACCACBBCABCBBCABBCBCCA
parte 1: 75
parte 2: 199727390721 con 3,298,534,883,329 letras
